<a href="https://colab.research.google.com/github/nithish526/coverage/blob/master/DNA_LSTM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.layers import BatchNormalization

In [ ]:
class MRNASequenceClassifier:
    def __init__(self, file_path):
        self.file_path = file_path
        self.dataset = None
        self.label_encoder = LabelEncoder()
        self.codon2idx = self._create_codon_dict()
        self.max_len = None
        self.vocab_size = len(self.codon2idx) + 1  # +1 for padding
        self.num_classes = None
        self.X_train = self.X_val = self.X_test = None
        self.y_train = self.y_val = self.y_test = None
        self.model = None

    # ------------------- Helper Methods -------------------
    def _create_codon_dict(self):
        nucleotides = ["A", "T", "G", "C"]
        codons = ["".join(p) for p in itertools.product(nucleotides, repeat=3)]
        codon2idx = {codon: i for i, codon in enumerate(codons)}
        return codon2idx

    def mount_drive(self):
        drive.mount('/content/drive', force_remount=True)

    # ------------------- Data Loading -------------------
    def load_data(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self.dataset = pd.read_excel(self.file_path)
        print("✅ Dataset loaded.")
        print(self.dataset.head(2))

    # ------------------- Preprocessing -------------------
    def preprocess(self):
        if self.dataset is None:
            raise ValueError("Dataset not loaded yet.")

        # Drop missing values
        self.dataset = self.dataset.dropna()

        # Encode target labels
        self.dataset['y_encoded'] = self.label_encoder.fit_transform(self.dataset['Organism'])
        self.num_classes = len(self.label_encoder.classes_)

        # Convert sequences to codons
        def split_and_join_codons(seq):
            codons = [seq[i:i+3] for i in range(0, len(seq)-2, 3)]
            return "-".join(codons)

        self.dataset['Codon_Sequence'] = [split_and_join_codons(seq) for seq in self.dataset["Sequence"].values]

        # Encode sequences using codon2idx
        encoded_sequences = [
            [self.codon2idx[c] for c in seq.split("-") if c in self.codon2idx]
            for seq in self.dataset["Codon_Sequence"].values
        ]

        # Pad sequences
        self.max_len = max(len(seq) for seq in encoded_sequences)
        X_padded = pad_sequences(encoded_sequences, maxlen=self.max_len, padding='post', value=0)

        # Convert labels to one-hot
        y_categorical = to_categorical(self.dataset['y_encoded'], num_classes=self.num_classes)

        # Train/Val/Test split
        X_temp, self.X_test, y_temp, self.y_test = train_test_split(
            X_padded, y_categorical, test_size=0.1, random_state=42, stratify=y_categorical
        )
        val_size = 0.1 / 0.9
        self.X_train, self.X_val, self.y_train, self.y_val = train_test_split(
            X_temp, y_temp, test_size=val_size, random_state=42, stratify=y_temp
        )

        print("✅ Preprocessing complete.")
        print(f"Training set: {self.X_train.shape}, {self.y_train.shape}")
        print(f"Validation set: {self.X_val.shape}, {self.y_val.shape}")
        print(f"Testing set: {self.X_test.shape}, {self.y_test.shape}")

    # ------------------- Model Building -------------------
    def build_model(self, embedding_dim=128, lstm_units=128, dense_units=256, dropout_rate=0.3):
        self.model = Sequential([
            # 🔹 Embedding Layer
            Embedding(input_dim=self.vocab_size, output_dim=embedding_dim, input_length=self.max_len),

            # 🔹 Deep BiLSTM Layers
            Bidirectional(LSTM(lstm_units, return_sequences=True)),
            Dropout(dropout_rate),
            BatchNormalization(),

            Bidirectional(LSTM(lstm_units // 2, return_sequences=False)),
            Dropout(dropout_rate),
            BatchNormalization(),

            # 🔹 Dense Layers (decreasing size)
            Dense(dense_units, activation='relu'),
            BatchNormalization(),
            Dropout(0.4),

            Dense(192, activation='relu'),
            BatchNormalization(),
            Dropout(0.4),

            Dense(128, activation='relu'),
            BatchNormalization(),
            Dropout(0.3),

            Dense(96, activation='relu'),
            Dense(64, activation='relu'),
            Dense(48, activation='relu'),
            Dense(32, activation='relu'),
            Dense(16, activation='relu'),

            # 🔹 Output Layer
            Dense(self.num_classes, activation='softmax')
        ])

        # 🔹 Compile model
        self.model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        print("✅ Deep BiLSTM model compiled successfully.")
        self.model.summary()

    # ------------------- Model Training -------------------
    def train(self, epochs=15, batch_size=32):
        if self.model is None:
            raise ValueError("Model not built yet.")
        history = self.model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_val, self.y_val),
            epochs=epochs,
            batch_size=batch_size
        )
        return history

    # ------------------- Evaluation -------------------
    def evaluate(self):
        if self.model is None:
            raise ValueError("Model not built yet.")
        loss, acc = self.model.evaluate(self.X_test, self.y_test)
        print(f"Test Loss: {loss:.4f}")
        print(f"Test Accuracy: {acc:.4f}")
        return loss, acc

    # ------------------- Prediction -------------------
    def predict_sequence(self, sequence):
        codon_seq = "-".join([sequence[i:i+3] for i in range(0, len(sequence)-2, 3)])
        encoded_seq = [self.codon2idx[c] for c in codon_seq.split("-") if c in self.codon2idx]
        X_input = pad_sequences([encoded_seq], maxlen=self.max_len, padding='post', value=0)
        pred = self.model.predict(X_input)
        class_idx = np.argmax(pred)
        organism = self.label_encoder.classes_[class_idx]
        return organism


In [ ]:
classifier = MRNASequenceClassifier("/content/drive/My Drive/DNA Project /dl dataset mrna .xlsx")
classifier.mount_drive()
classifier.load_data()
classifier.preprocess()
classifier.build_model()
classifier.train(epochs=20, batch_size=64)
classifier.evaluate()



Mounted at /content/drive
✅ Dataset loaded.
           ID                Organism  \
0  AF536133.1  Haemophilus influenzae   
1  AJ418051.1      Pseudomonas putida   

                                         Description  Sequence_Length  \
0  Haemophilus influenzae isolate 486 glucose-6-p...              469   
1  Pseudomonas putida partial merA gene for putat...             1558   

                                            Sequence  Frequency  
0  AATCGCACTAATACGCCAGTGCTTGTTGATGGCAAAGATGTCATGC...        961  
1  ATGACCCATCTAAAAATCACCGGCATGACCTGCGACTCGTGCGCGG...        692  
✅ Preprocessing complete.
Training set: (41841, 66), (41841, 62)
Validation set: (5231, 66), (5231, 62)
Testing set: (5231, 66), (5231, 62)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


✅ Deep BiLSTM model compiled successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 33s 29ms/step - accuracy: 0.1535 - loss: 3.5848 - val_accuracy: 0.3215 - val_loss: 2.5843
Epoch 2/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 18s 27ms/step - accuracy: 0.3238 - loss: 2.6172 - val_accuracy: 0.3705 - val_loss: 2.3707
Epoch 3/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 19s 29ms/step - accuracy: 0.3848 - loss: 2.3238 - val_accuracy: 0.4760 - val_loss: 1.9771
Epoch 4/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.4439 - loss: 2.1079 - val_accuracy: 0.4976 - val_loss: 1.8248
Epoch 5/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.4789 - loss: 1.9563 - val_accuracy: 0.5420 - val_loss: 1.6706
Epoch 6/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.5093 - loss: 1.8364 - val_accuracy: 0.5752 - val_loss: 1.5495
Epoch 7/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 18s 27ms/step - accuracy: 0.5398 - loss: 1.7129 - val_accuracy: 0.5957 - val_loss: 1.4683
Epoch 8/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 19s 29ms/step - accuracy: 0.5616 - loss: 1.6136 - 

(1.0277173519134521, 0.7208946943283081)

In [ ]:

sample_sequence =str(input("Enter the sequence: "))
# Make prediction
predicted_organism = classifier.predict_sequence(sample_sequence)

print(f"Predicted organism for the sequence: {predicted_organism}")


Enter the sequence: AAGTAAAATAGCAAAGGATTGTGAAGAAAATATAATAGTTTTTTGTGGGGTTAAATTTATGGCAGAAAGTGCAA
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Predicted organism for the sequence: Enterococcus faecium


In [ ]:
# Save the entire model (architecture + weights + optimizer state)
model_save_path = "/content/drive/MyDrive/mRNA_Project/model_bilstm.h5"
classifier.model.save(model_save_path)
import joblib
joblib.dump(classifier.label_encoder, "/content/drive/MyDrive/mRNA_Project/label_encoder.pkl")
print(f"✅ Model saved to: {model_save_path}")


✅ Model saved to: /content/drive/MyDrive/mRNA_Project/model_bilstm.h5


In [ ]:

sample_sequence =str(input("Enter the sequence: "))
# Make prediction
predicted_organism = classifier.predict_sequence(sample_sequence)

print(f"Predicted organism for the sequence: {predicted_organism}")


Enter the sequence: GGTCATCATGCTTTCCGCAACCGCACAGACGCTGCATAAGTTTTTTTAGTATATTCATGTCATTCTCCTGTTCTGCCTGTATCACTGCCCACTTCATCCAGCCCCTTAACATCCTGCCACGGCCCGTCACCAAACCTGACCTGCAAATGCTGAAAAAAACCCTGAACCCGTGTGGCATCTTTGGGGGCAAGAAAGGTCAG
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Predicted organism for the sequence: Escherichia coli


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/mRNA_Project/model_bilstm.h5")
from google.colab import files
files.download("/content/drive/MyDrive/mRNA_Project/label_encoder.pkl")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>